# Дообучение эмбеддинговой модели на банковских документах

**Модель:** `deepvk/USER-bge-m3`  
**Задача:** улучшение качества retriever в RAG-системе для банковских документов

## Содержание
1. Установка и настройка окружения
2. Подготовка данных и формирование триплетов
3. Конфигурация и запуск обучения (3 эксперимента)
4. Оценка качества: оригинальная vs дообученные модели
5. Визуализация и выводы

## 0. Установка зависимостей

In [ ]:
# Раскомментируйте при первом запуске
# !pip install -r ../requirements.txt

In [ ]:
import sys
import os
import logging
import warnings

# Добавляем корень проекта в path
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src import (
    Chunk,
    TripletSampler,
    BM25HardNegativeMiner,
    generate_sample_banking_chunks,
    EmbeddingTrainer,
    TrainingConfig,
    LossType,
    RetrievalEvaluator,
    RetrievalMetrics,
)
from src.metrics import build_evaluation_set
from src.trainer import make_cpu_config, make_gpu_config
from src.data_preparation import DocumentChunker

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
warnings.filterwarnings("ignore")

# Определяем устройство
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 1. Подготовка данных и формирование триплетов

### 1.1 Загрузка / генерация банковских чанков

В реальном проекте здесь будет загрузка ваших документов.
Для демонстрации используем встроенный синтетический корпус.

In [ ]:
# ── Вариант А: синтетические данные (демо) ─────────────────────────────────
chunks = generate_sample_banking_chunks()

# ── Вариант Б: ваши реальные документы ────────────────────────────────────
# from src.data_preparation import DocumentChunker
# chunker = DocumentChunker(chunk_size=256, overlap=32, split_by="paragraph")
# chunks = []
# for path in Path("../data/").glob("*.txt"):
#     text = path.read_text(encoding="utf-8")
#     doc_chunks = chunker.chunk_document(text, doc_id=path.stem, section="")
#     chunks.extend(doc_chunks)

print(f"Всего чанков: {len(chunks)}")
print(f"Документов:   {len(set(c.doc_id for c in chunks))}")
print(f"Секций:       {len(set(c.section for c in chunks))}")

# Показываем несколько чанков
pd.DataFrame([
    {"doc_id": c.doc_id, "section": c.section, "chunk_id": c.chunk_id, 
     "text_preview": c.text[:100] + "..."}
    for c in chunks
]).head(10)

### 1.2 Формирование триплетов (anchor, positive, negative)

| Стратегия | Anchor | Positive | Negative |
|-----------|--------|----------|----------|
| `adjacent` | чанк_i | чанк_{i±1} того же документа | BM25-hard из другого документа |
| `section` | чанк из секции X | другой чанк из секции X | чанк из другой секции |
| `query_template` | шаблонный запрос | исходный чанк | BM25-hard из другого документа |

In [ ]:
sampler = TripletSampler(
    chunks=chunks,
    negative_strategy="bm25",  # "bm25" | "random" | "mixed"
    n_hard_neg_pool=10,
    seed=42,
)

triplets_df = sampler.to_dataframe(
    strategies=["adjacent", "section", "query_template"],
    n_per_chunk=2,
    deduplicate=True,
)

print(f"Итого триплетов: {len(triplets_df)}")
print("\nРаспределение по стратегиям:")
print(triplets_df["strategy"].value_counts())

In [ ]:
# Просматриваем DataFrame
pd.set_option("display.max_colwidth", 80)
triplets_df[["anchor", "positive", "negative", "strategy"]].head(10)

In [ ]:
# Анализ длин текстов
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ["anchor", "positive", "negative"], ["steelblue", "forestgreen", "tomato"]):
    lengths = triplets_df[col].str.split().str.len()
    ax.hist(lengths, bins=30, color=color, edgecolor="white", alpha=0.85)
    ax.set_title(f"{col}: длина (слов)", fontsize=12)
    ax.set_xlabel("слов")
    ax.set_ylabel("частота")
    ax.axvline(lengths.median(), color="black", linestyle="--", label=f"median={lengths.median():.0f}")
    ax.legend()
plt.suptitle("Распределение длин anchor / positive / negative", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 1.3 Train / Eval разбивка

In [ ]:
from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(triplets_df, test_size=0.15, random_state=42, shuffle=True)
train_df = train_df.reset_index(drop=True)
eval_df  = eval_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Eval: {len(eval_df)}")

---
## 2. Оценочный датасет (для сравнения моделей)

Формируем `queries`, `corpus`, `qrels` из триплетов.

In [ ]:
queries, corpus, qrels = build_evaluation_set(
    triplets_df=triplets_df,
    n_queries=min(150, len(triplets_df)),
    seed=42,
)

evaluator = RetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    qrels=qrels,
    ks=(1, 3, 5, 10),
)

print(f"Запросов: {len(queries)} | Корпус: {len(corpus)} документов")

---
## 3. Эксперимент 0: Baseline (оригинальная модель)

Оцениваем `deepvk/USER-bge-m3` без дообучения — это наш baseline.

In [ ]:
MODEL_NAME = "deepvk/USER-bge-m3"

print("Оцениваем оригинальную модель...")
baseline_metrics = evaluator.evaluate(MODEL_NAME, batch_size=32)

pd.DataFrame([baseline_metrics.to_dict()])

---
## 4. Эксперимент 1: TripletLoss

**Формула:** `L = max(0, d(a,p) − d(a,n) + margin)`

- Прямолинейный, интерпретируемый
- Требует явных триплетов (anchor, positive, negative)
- Медленнее MNRL (нет in-batch negatives)

In [ ]:
if DEVICE == "cpu":
    config_triplet = make_cpu_config(
        loss_type=LossType.TRIPLET,
        num_epochs=2,
        output_dir="../output/exp1_triplet",
    )
else:
    config_triplet = make_gpu_config(
        loss_type=LossType.TRIPLET,
        num_epochs=3,
        output_dir="../output/exp1_triplet",
    )

config_triplet.triplet_margin = 0.5
print(config_triplet)

In [ ]:
trainer_triplet = EmbeddingTrainer(config_triplet)
trainer_triplet.train(train_df, eval_df)
trainer_triplet.save_model("../output/exp1_triplet/final")
print("Эксперимент 1 завершён.")

In [ ]:
metrics_triplet = evaluator.evaluate("../output/exp1_triplet/final", batch_size=32)
pd.DataFrame([metrics_triplet.to_dict()])

---
## 5. Эксперимент 2: MultipleNegativesRankingLoss (MNRL)

**Формула:** `L = -log(exp(sim(a_i, p_i)/τ) / Σ_j exp(sim(a_i, p_j)/τ))`

- In-batch negatives — каждый positive других запросов в батче становится негативом
- **Рекомендован как основной** — хорошо масштабируется с размером батча
- Работает с парами (anchor, positive) — не нужны явные негативы
- Если есть колонка `negative` — она добавляется как дополнительный hard negative

In [ ]:
if DEVICE == "cpu":
    config_mnrl = make_cpu_config(
        loss_type=LossType.MNRL,
        num_epochs=2,
        output_dir="../output/exp2_mnrl",
    )
else:
    config_mnrl = make_gpu_config(
        loss_type=LossType.MNRL,
        num_epochs=3,
        output_dir="../output/exp2_mnrl",
    )

config_mnrl.temperature = 0.05
print(config_mnrl)

In [ ]:
trainer_mnrl = EmbeddingTrainer(config_mnrl)
trainer_mnrl.train(train_df, eval_df)
trainer_mnrl.save_model("../output/exp2_mnrl/final")
print("Эксперимент 2 завершён.")

In [ ]:
metrics_mnrl = evaluator.evaluate("../output/exp2_mnrl/final", batch_size=32)
pd.DataFrame([metrics_mnrl.to_dict()])

---
## 6. Эксперимент 3: TSDAE (Unsupervised)

**Идея:** Corrupted(sentence) → Encoder → Decoder → Original(sentence)

- Не требует разметки (только сырые тексты)
- Адаптирует модель к доменной лексике банковских документов
- Можно использовать как **pre-fine-tuning** перед MNRL

In [ ]:
# Для TSDAE нужны только тексты (без разметки)
all_texts = [c.text for c in chunks]
tsdae_df = pd.DataFrame({"anchor": all_texts})
print(f"Текстов для TSDAE: {len(tsdae_df)}")

if DEVICE == "cpu":
    config_tsdae = make_cpu_config(
        loss_type=LossType.TSDAE,
        num_epochs=1,
        output_dir="../output/exp3_tsdae",
    )
else:
    config_tsdae = make_gpu_config(
        loss_type=LossType.TSDAE,
        num_epochs=2,
        output_dir="../output/exp3_tsdae",
    )

config_tsdae.tsdae_noise_ratio = 0.6
print(config_tsdae)

In [ ]:
trainer_tsdae = EmbeddingTrainer(config_tsdae)
trainer_tsdae.train(tsdae_df)  # eval не нужен для TSDAE
trainer_tsdae.save_model("../output/exp3_tsdae/final")
print("Эксперимент 3 (TSDAE) завершён.")

In [ ]:
metrics_tsdae = evaluator.evaluate("../output/exp3_tsdae/final", batch_size=32)
pd.DataFrame([metrics_tsdae.to_dict()])

---
## 7. Сравнение всех моделей

In [ ]:
# Собираем все результаты
all_results = [
    baseline_metrics.to_dict(),
    metrics_triplet.to_dict(),
    metrics_mnrl.to_dict(),
    metrics_tsdae.to_dict(),
]

comparison_df = pd.DataFrame(all_results)
comparison_df.insert(0, "alias", ["baseline", "triplet", "mnrl", "tsdae"])
comparison_df = comparison_df.set_index("alias")

# Выделяем лучшие значения по каждой метрике
metric_cols = ["Recall@1", "Recall@3", "Recall@5", "Recall@10", "MRR@10", "NDCG@10", "MAP@10"]
styled = comparison_df[metric_cols].style.highlight_max(axis=0, color="lightgreen").highlight_min(axis=0, color="#ffcccc")
styled

In [ ]:
# Дельта относительно baseline
delta_df = comparison_df[metric_cols].subtract(comparison_df.loc["baseline", metric_cols]).round(4)
delta_df = delta_df.drop(index="baseline")
print("Прирост метрик относительно baseline:")
delta_df.style.background_gradient(cmap="RdYlGn", axis=None, vmin=-0.1, vmax=0.1)

In [ ]:
# ── График 1: Recall@k сравнение ─────────────────────────────────────────
recall_cols = ["Recall@1", "Recall@3", "Recall@5", "Recall@10"]
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(recall_cols))
width = 0.2
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

for i, (alias, row) in enumerate(comparison_df[recall_cols].iterrows()):
    bars = ax.bar(x + i * width, row.values, width, label=alias, color=colors[i], edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", fontsize=7, padding=2)

ax.set_title("Recall@k: Baseline vs Дообученные модели", fontsize=14, fontweight="bold")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([c.replace("Recall@", "@") for c in recall_cols])
ax.set_ylabel("Recall", fontsize=12)
ax.set_ylim(0, 1.15)
ax.legend(title="Модель", fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("../output/recall_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── График 2: MRR / NDCG / MAP ────────────────────────────────────────────
other_cols = ["MRR@10", "NDCG@10", "MAP@10"]
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(other_cols))
for i, (alias, row) in enumerate(comparison_df[other_cols].iterrows()):
    bars = ax.bar(x + i * width, row.values, width, label=alias, color=colors[i], edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", fontsize=7, padding=2)

ax.set_title("MRR@10 / NDCG@10 / MAP@10", fontsize=14, fontweight="bold")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(other_cols)
ax.set_ylabel("Score", fontsize=12)
ax.set_ylim(0, 1.15)
ax.legend(title="Модель", fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("../output/mrr_ndcg_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── График 3: Radar chart ─────────────────────────────────────────────────
metrics_for_radar = ["Recall@1", "Recall@5", "Recall@10", "MRR@10", "NDCG@10", "MAP@10"]
N = len(metrics_for_radar)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for i, (alias, row) in enumerate(comparison_df[metrics_for_radar].iterrows()):
    values = row.tolist() + [row.iloc[0]]
    ax.plot(angles, values, linewidth=2, linestyle="solid", label=alias, color=colors[i])
    ax.fill(angles, values, alpha=0.15, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_for_radar, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title("Профиль качества моделей", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.15), fontsize=10)
plt.tight_layout()
plt.savefig("../output/radar_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. Качественный анализ: примеры retrieval

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def show_retrieval_examples(model_path: str, queries_sample: list, corpus_texts: list, top_k: int = 3):
    model = SentenceTransformer(model_path)
    corpus_embs = model.encode(corpus_texts, normalize_embeddings=True, show_progress_bar=False)
    
    for query in queries_sample:
        q_emb = model.encode([query], normalize_embeddings=True)
        sims = cosine_similarity(q_emb, corpus_embs)[0]
        top_idx = np.argsort(sims)[::-1][:top_k]
        print(f"\n🔍 Запрос: {query}")
        for rank, idx in enumerate(top_idx, 1):
            print(f"  [{rank}] ({sims[idx]:.3f}) {corpus_texts[idx][:120]}...")

sample_queries = [
    "Что такое льготный период по кредитной карте?",
    "Какой минимальный первоначальный взнос по ипотеке?",
    "Как застрахованы вклады в банке?",
]

corpus_texts_list = [c.text for c in chunks]

print("=" * 70)
print("BASELINE")
print("=" * 70)
show_retrieval_examples(MODEL_NAME, sample_queries, corpus_texts_list)

print("\n" + "=" * 70)
print("MNRL (дообученная)")
print("=" * 70)
show_retrieval_examples("../output/exp2_mnrl/final", sample_queries, corpus_texts_list)

---
## 9. Итоги и рекомендации

### Выводы по экспериментам

| Эксперимент | Loss | Сложность | Данные | Рекомендация |
|-------------|------|-----------|--------|---------------|
| Baseline | — | — | не нужны | отправная точка |
| Exp 1 | TripletLoss | средняя | anchor+pos+neg | хорош при малых данных |
| **Exp 2** | **MNRL** | **низкая** | **anchor+pos** | **рекомендован** |
| Exp 3 | TSDAE | высокая | только тексты | unsupervised pre-finetune |

### Следующие шаги
1. **Увеличить корпус**: добавить все реальные банковские документы
2. **Синтетические Q&A**: использовать LLM (GPT/Qwen) для генерации пар вопрос-ответ
3. **Hard negative mining**: после 1-й эпохи обучения заменить BM25-негативы на embedding-hard
4. **TSDAE → MNRL**: двухэтапное обучение: сначала TSDAE (доменная адаптация), затем MNRL
5. **MatryoshkaLoss**: обернуть MNRL в MatryoshkaLoss для поддержки разных размерностей эмбеддингов

In [ ]:
# Финальная таблица результатов
print("\n" + "=" * 70)
print("ФИНАЛЬНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("=" * 70)
print(comparison_df[metric_cols].to_string())
print("\n")

best_model_alias = comparison_df["Recall@10"].idxmax()
best_recall = comparison_df.loc[best_model_alias, "Recall@10"]
baseline_recall = comparison_df.loc["baseline", "Recall@10"]
improvement = (best_recall - baseline_recall) / baseline_recall * 100

print(f"✅ Лучшая модель: {best_model_alias}")
print(f"   Recall@10: {best_recall:.4f} vs baseline {baseline_recall:.4f}")
print(f"   Прирост: +{improvement:.1f}%")